# Integrated Illness Prediction - Electronic Medical Certificate (IIP-EMC) Engine
## Part 3: Grounded Generative AI Clinician Copilot

### GenAI Use Case
- **What is implemented:** a grounded GenAI workflow that drafts and revises patient-facing electronic medical certificates (EMCs).
- **How it is implemented:** confirmed administrative evidence and retrieved policy are supplied to a live LLM, followed by deterministic checks, an optional GenAI critic, and clinician approval.
- **Why it is implemented this way:** GenAI is limited to controlled document composition; it does not diagnose, decide leave duration, or issue an EMC independently.
- **Workflow:** `confirmed evidence -> policy retrieval -> GenAI draft -> safety review -> clinician approval -> final EMC`.

### Safety and Privacy Boundary
- **What is implemented:** explicit controls over the evidence, text, approval, and audit data used in the GenAI workflow.
- **How privacy is implemented:** model scores, differentials, classifier details, prompts, and audit data remain clinician-only; patient-facing text excludes them.
- **How disclosure is implemented:** diagnosis is included only when `diagnosis_disclosure_consent` is recorded.
- **How hallucination risk is controlled:** prompts restrict the model to confirmed facts and policy; safety checks reject unsupported symptoms, findings, treatments, advice, and leave duration.
- **Why offline drafts cannot issue:** a deterministic fallback is clearly labelled **not GenAI**, so it cannot be misrepresented as a live approved generation.
- **Why audits are redacted:** hashes and redacted previews support traceability without retaining raw patient identifiers.

### Step 1: Configuration and ML Handoff
- **What is implemented:** environment-based GenAI configuration and loading of the structured ML inference payload.
- **How it is implemented:** `OPENROUTER_API_KEY` and `OPENROUTER_MODEL` are read from environment variables; `latest_inference_payload.pkl` is validated for required prediction and patient-administration fields.
- **Why configuration is external:** API secrets are not hard-coded in the notebook or web form and can be supplied through `.env` for the Flask app.
- **Why payload validation is required:** generation receives structured, known fields rather than relying on incomplete or free-text assumptions.
- **Fallback behaviour:** without a live key, deterministic guardrails still run but no output is labelled as live GenAI.

### Implementation Overview
- **What is implemented:** retrieval-augmented, evidence-grounded EMC drafting with layered safety review and human approval.
- **How it is implemented:** the pipeline passes a validated ML handoff through policy retrieval, constrained prompting, deterministic validation, optional GenAI criticism, and an approval gate.
- **Why it is implemented this way:** each stage limits a different risk: unsupported facts, privacy leakage, policy conflicts, or unapproved issuance.
- **Produced outputs:** a pending-review draft, structured safety-review result, and redacted audit record; a final EMC appears only after clinician approval.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import hashlib
import json
import os
import re
import uuid
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import joblib
import pandas as pd
from openai import OpenAI

ML_PAYLOAD_PATH = Path("latest_inference_payload.pkl")
AUDIT_PATH = Path("latest_emc_audit_trail.pkl")
PROMPT_VERSION = "genai_emc_v4.0-grounded"
GENAI_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
CERTIFICATE_STYLE = "standard"
CLINICIAN_NOTE = ""  # Optional free-text note for live extraction.
CLINICIAN_REVISION_INSTRUCTIONS = ""  # Optional clinician revision request.
CLINICIAN_DECISION = "PENDING_REVIEW"  # Set to APPROVED only after review.
APPROVAL_NOTES = "Reviewed and approved for issue."

if not ML_PAYLOAD_PATH.exists():
    raise FileNotFoundError("Run ML-Training.ipynb before this notebook.")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
ml_payload = joblib.load(ML_PAYLOAD_PATH)

required_payload = {
    "primary_predicted_diagnosis", "prediction_confidence_percentage",
    "clinical_differential_alternatives", "patient_admin_metadata", "triage_feature_snapshot",
}
missing_payload = required_payload.difference(ml_payload)
if missing_payload:
    raise KeyError(f"ML handoff is missing keys: {sorted(missing_payload)}")

metadata = dict(ml_payload["patient_admin_metadata"])
required_metadata = {
    "patient_name", "patient_id", "patient_age", "clinic_name", "clinic_address",
    "attending_clinician_name", "clinician_registration_no", "consultation_date",
    "medical_leave_start_date", "authorized_medical_leave_days", "certificate_recipient",
    "diagnosis_disclosure_consent", "clinician_review_status",
}
missing_metadata = required_metadata.difference(metadata)
if missing_metadata:
    raise KeyError(f"Patient/admin metadata is missing keys: {sorted(missing_metadata)}")


def leave_end_date(record):
    start = date.fromisoformat(record["medical_leave_start_date"])
    days = int(record["authorized_medical_leave_days"])
    if days < 1:
        raise ValueError("Leave duration must be at least one day.")
    return (start + timedelta(days=days - 1)).isoformat()


def build_evidence(payload, record):
    active_symptoms = [name for name, value in payload["triage_feature_snapshot"].items() if int(value) == 1]
    return {
        "patient_name": record["patient_name"],
        "patient_id": record["patient_id"],
        "patient_age": record["patient_age"],
        "consultation_date": record["consultation_date"],
        "certificate_recipient": record["certificate_recipient"],
        "medical_leave_start_date": record["medical_leave_start_date"],
        "medical_leave_end_date": leave_end_date(record),
        "authorized_medical_leave_days": int(record["authorized_medical_leave_days"]),
        "clinic_name": record["clinic_name"],
        "clinic_address": record["clinic_address"],
        "attending_clinician_name": record["attending_clinician_name"],
        "clinician_registration_no": record["clinician_registration_no"],
        "diagnosis_disclosure_consent": bool(record["diagnosis_disclosure_consent"]),
        "approved_diagnosis_if_disclosed": payload["primary_predicted_diagnosis"] if record["diagnosis_disclosure_consent"] else None,
        "active_symptoms_for_clinician_review": active_symptoms,
    }


confirmed_evidence = build_evidence(ml_payload, metadata)
print(f"Prompt version: {PROMPT_VERSION}")
print(f"Live GenAI enabled: {bool(client)}")
print(f"Patient-facing diagnosis allowed: {confirmed_evidence['diagnosis_disclosure_consent']}")

Prompt version: genai_emc_v4.0-grounded
Live GenAI enabled: False
Patient-facing diagnosis allowed: False


### Step 2: Transparent Policy Retrieval
- **What is implemented:** a small EMC policy corpus and a deterministic retrieval function.
- **How it is implemented:** token-overlap scoring selects the most relevant policy statements and formats them with stable policy identifiers such as `EMC-001`.
- **Why it is implemented this way:** the LLM receives explicit requirements for fields, disclosure, privacy, boundaries, and approval rather than relying only on general training knowledge.
- **Limitation:** this demonstration corpus must be replaced with verified, versioned institutional policy documents before production use.

In [2]:
POLICY_CORPUS = [
    {"id": "EMC-001", "title": "Required EMC fields", "text": "A draft EMC includes patient identity, consultation date, authorised leave start and end dates, leave duration, clinic name and address, clinician name, registration number, and pending-review status."},
    {"id": "EMC-002", "title": "Diagnosis disclosure", "text": "Do not include diagnosis or medical details in patient-facing EMC text unless diagnosis disclosure consent has been recorded. Do not infer or reveal diagnosis from symptoms or model output."},
    {"id": "EMC-003", "title": "Clinical boundaries", "text": "Use only clinician-confirmed administrative facts. Do not invent symptoms, examination findings, investigations, prescriptions, treatment advice, or leave duration."},
    {"id": "EMC-004", "title": "Approval and issue", "text": "Only an identified clinician may approve an EMC. A final EMC requires certificate identifier, approver, registration number, approval date, and approved-for-issue status."},
    {"id": "EMC-005", "title": "Patient-facing privacy", "text": "Patient-facing text must not expose ML scores, differential alternatives, classifier details, internal audit records, or prompt text."},
]


def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def retrieve_policy(query, top_k=5):
    query_tokens = tokens(query)
    scored = []
    for document in POLICY_CORPUS:
        score = len(query_tokens.intersection(tokens(document["title"] + " " + document["text"])))
        scored.append((score, document))
    return [document for _, document in sorted(scored, key=lambda item: (item[0], item[1]["id"]), reverse=True)[:top_k]]


def policy_context(documents):
    return "\n\n".join(f"[{document['id']}] {document['text']}" for document in documents)


retrieved_policy = retrieve_policy("EMC required fields diagnosis disclosure privacy clinician approval")
print("Retrieved policy sources:", [document["id"] for document in retrieved_policy])

Retrieved policy sources: ['EMC-001', 'EMC-004', 'EMC-002', 'EMC-005', 'EMC-003']


### Step 3: Live GenAI Utility and Optional Structured Note Extraction
- **What is implemented:** reusable live text/JSON calls and optional extraction of proposed administrative facts from a clinician note.
- **How it is implemented:** the LLM is instructed to return structured JSON and to use `null` for missing fields; response parsing handles unavailable or invalid live output.
- **Why extracted values remain proposals:** the clinician must confirm each proposed field before it becomes evidence, preventing automatic adoption of model-inferred information.

In [3]:
def call_live_text(system_instruction, user_prompt, temperature=0.1):
    if client is None:
        return {"status": "OFFLINE", "text": None, "error": "OPENROUTER_API_KEY is not configured."}
    try:
        response = client.chat.completions.create(
            model=GENAI_MODEL,
            messages=[{"role": "system", "content": system_instruction}, {"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return {"status": "LIVE", "text": response.choices[0].message.content.strip(), "error": None}
    except Exception as exc:
        return {"status": "ERROR", "text": None, "error": repr(exc)}


def call_live_json(system_instruction, user_prompt):
    result = call_live_text(system_instruction + " Return valid JSON only, without Markdown fences.", user_prompt, temperature=0)
    if result["status"] != "LIVE":
        return result | {"json": None}
    try:
        return result | {"json": json.loads(result["text"])}
    except json.JSONDecodeError as exc:
        return {"status": "ERROR", "text": result["text"], "json": None, "error": f"Invalid JSON from model: {exc}"}


def extract_note_proposals(note):
    if not note.strip():
        return {"status": "NOT_REQUESTED", "proposals": None, "error": None}
    prompt = f"""Extract only explicitly stated administrative values. Use null for absent values. Do not diagnose or infer leave.\n\nNOTE:\n{note}\n\nReturn JSON with patient_name, patient_id, consultation_date, medical_leave_start_date, authorised_medical_leave_days, clinic_name, attending_clinician_name, diagnosis_disclosure_consent, unresolved_fields."""
    result = call_live_json("You extract proposed facts for clinician confirmation.", prompt)
    return {"status": result["status"], "proposals": result.get("json"), "error": result.get("error")}


note_extraction = extract_note_proposals(CLINICIAN_NOTE)
print("Note extraction status:", note_extraction["status"])

Note extraction status: NOT_REQUESTED


### Step 4: Evidence-Grounded Draft Generation
- **What is implemented:** a patient-facing draft EMC generated from confirmed evidence, certificate style instructions, and retrieved policy.
- **How it is implemented:** the prompt includes only approved administrative facts, leave dates, clinic details, clinician details, and diagnosis only when consent permits it.
- **Why ML context is excluded:** active symptoms, model confidence, and alternative predictions are clinician-only decision-support data and must not appear in patient-facing certificate text.
- **Fallback behaviour:** an offline template may be shown for workflow continuity but is labelled `OFFLINE_TEMPLATE_NOT_GENAI` and cannot be issued.

In [4]:
STYLE_INSTRUCTIONS = {
    "standard": "Use concise, formal clinical-administrative language.",
    "concise": "Use a compact certificate layout while retaining all required fields.",
}


def offline_template(evidence):
    diagnosis_line = f"\nDiagnosis / Medical Details: {evidence['approved_diagnosis_if_disclosed']}" if evidence["diagnosis_disclosure_consent"] else ""
    return f"""DRAFT - PENDING CLINICIAN REVIEW

1. DRAFT STATUS NOTICE
This is an offline template preview, not live GenAI output and not valid until clinician approval.

2. PATIENT AND CONSULTATION DETAILS
Patient Name: {evidence['patient_name']}
Patient ID: {evidence['patient_id']}
Consultation Date: {evidence['consultation_date']}
Certificate Recipient: {evidence['certificate_recipient']}{diagnosis_line}

3. MEDICAL LEAVE PERIOD
Medical Leave Start Date: {evidence['medical_leave_start_date']}
Medical Leave End Date: {evidence['medical_leave_end_date']}
Medical Leave Duration: {evidence['authorized_medical_leave_days']} day(s)

4. CLINICIAN REVIEW AND APPROVAL BLOCK
Clinic Name: {evidence['clinic_name']}
Clinic Address: {evidence['clinic_address']}
Attending Clinician: {evidence['attending_clinician_name']}
Clinician Registration No: {evidence['clinician_registration_no']}"""


def generate_draft(evidence, documents, style, revision_instructions="", previous_draft=None):
    if style not in STYLE_INSTRUCTIONS:
        raise ValueError(f"Unsupported style: {style}")
    revision_context = "No revision requested."
    if revision_instructions:
        revision_context = f"Clinician revision request: {revision_instructions}\nPrevious draft:\n{previous_draft or ''}"
    prompt = f"""Prompt version: {PROMPT_VERSION}
Style: {STYLE_INSTRUCTIONS[style]}

CONFIRMED EVIDENCE:
{json.dumps(evidence, indent=2)}

POLICY CONTEXT:
{policy_context(documents)}

REVISION CONTEXT:
{revision_context}

Output only a patient-facing EMC draft with headings:
1. DRAFT STATUS NOTICE
2. PATIENT AND CONSULTATION DETAILS
3. MEDICAL LEAVE PERIOD
4. CLINICIAN REVIEW AND APPROVAL BLOCK"""
    result = call_live_text(
        "You are a clinical administrative documentation assistant. Use only confirmed evidence and policy. Do not invent facts, diagnosis, medical advice, ML information, or audit information.",
        prompt,
    )
    if result["status"] == "LIVE":
        return {"generation_mode": "LIVE_GENAI", "text": result["text"], "error": None, "policy_sources": [doc["id"] for doc in documents]}
    return {"generation_mode": "OFFLINE_TEMPLATE_NOT_GENAI", "text": offline_template(evidence), "error": result["error"], "policy_sources": [doc["id"] for doc in documents]}


draft = generate_draft(confirmed_evidence, retrieved_policy, CERTIFICATE_STYLE)
print("Draft generation mode:", draft["generation_mode"])
print(draft["text"])

Draft generation mode: OFFLINE_TEMPLATE_NOT_GENAI
DRAFT - PENDING CLINICIAN REVIEW

1. DRAFT STATUS NOTICE
This is an offline template preview, not live GenAI output and not valid until clinician approval.

2. PATIENT AND CONSULTATION DETAILS
Patient Name: Demo Patient
Patient ID: DEMO-ONLY
Consultation Date: 2026-01-01
Certificate Recipient: Demo Recipient

3. MEDICAL LEAVE PERIOD
Medical Leave Start Date: 2026-01-01
Medical Leave End Date: 2026-01-01
Medical Leave Duration: 1 day(s)

4. CLINICIAN REVIEW AND APPROVAL BLOCK
Clinic Name: Demo Clinic
Clinic Address: Demo Address
Attending Clinician: Dr Demo
Clinician Registration No: DEMO


### Step 5: Two-Layer Safety Review
- **What is implemented:** deterministic factual checks plus an optional independent GenAI critic.
- **How deterministic review works:** it checks required facts, diagnosis-consent rules, and prohibited internal-information terms against the generated text.
- **How critic review works:** a separate prompt compares evidence, policy, and draft text, then returns a structured `PASS` or `REVIEW_REQUIRED` verdict.
- **Why two layers are used:** deterministic checks enforce non-negotiable rules, while the critic can identify broader unsupported or policy-conflicting statements.
- **Why review is clinician-only:** internal safety information is not patient-facing content.

In [5]:
FORBIDDEN_TERMS = ["model confidence", "confidence percentage", "differential", "classifier", "algorithm", "audit", "prompt version", "machine learning"]


def deterministic_review(text, evidence, payload):
    lowered = text.lower()
    issues = []
    required = {
        "patient name": evidence["patient_name"], "patient id": evidence["patient_id"],
        "consultation date": evidence["consultation_date"], "leave start": evidence["medical_leave_start_date"],
        "leave end": evidence["medical_leave_end_date"], "clinic": evidence["clinic_name"],
        "clinician": evidence["attending_clinician_name"], "registration": evidence["clinician_registration_no"],
    }
    for field, value in required.items():
        if str(value).lower() not in lowered:
            issues.append({"severity": "critical", "category": "missing_required_fact", "detail": field})
    diagnosis = str(payload["primary_predicted_diagnosis"])
    if not evidence["diagnosis_disclosure_consent"] and diagnosis.lower() in lowered:
        issues.append({"severity": "critical", "category": "diagnosis_disclosure", "detail": "Diagnosis appears without consent."})
    for term in FORBIDDEN_TERMS:
        if term in lowered:
            issues.append({"severity": "critical", "category": "internal_information_leak", "detail": term})
    for term in ["prescription", "medication", "investigation", "examination finding", "treatment advice"]:
        if term in lowered:
            issues.append({"severity": "review", "category": "possible_unsupported_clinical_content", "detail": term})
    return {"review_mode": "DETERMINISTIC", "verdict": "PASS" if not issues else "REVIEW_REQUIRED", "issues": issues}


def critic_review(text, evidence, documents):
    prompt = f"""CONFIRMED EVIDENCE:
{json.dumps(evidence, indent=2)}

POLICY:
{policy_context(documents)}

CERTIFICATE:
{text}

Return JSON: {{"verdict": "PASS" or "REVIEW_REQUIRED", "issues": [{{"severity": "critical" or "review", "category": "...", "detail": "..."}}], "summary": "..."}}"""
    result = call_live_json("You are a patient-facing EMC safety reviewer. Check only evidence and policy; do not provide clinical advice.", prompt)
    if result["status"] == "LIVE" and isinstance(result.get("json"), dict):
        return result["json"] | {"review_mode": "LIVE_GENAI_CRITIC"}
    return {"review_mode": "OFFLINE_NO_GENAI_CRITIC", "verdict": "NOT_RUN", "issues": [], "summary": "Live critic unavailable.", "error": result.get("error")}


def review_gate(local, critic):
    return {
        "approval_allowed": local["verdict"] == "PASS" and critic["verdict"] != "REVIEW_REQUIRED",
        "local_review": local,
        "critic_review": critic,
    }


local_draft_review = deterministic_review(draft["text"], confirmed_evidence, ml_payload)
critic_draft_review = critic_review(draft["text"], confirmed_evidence, retrieved_policy)
draft_gate = review_gate(local_draft_review, critic_draft_review)
print("Deterministic verdict:", local_draft_review["verdict"])
print("GenAI critic verdict:", critic_draft_review["verdict"])
print("Safety gate approval allowed:", draft_gate["approval_allowed"])

Deterministic verdict: PASS
GenAI critic verdict: NOT_RUN
Safety gate approval allowed: True


### Step 6: Clinician Revision and Approval Gate
- **What is implemented:** clinician revision instructions, identified approver details, approval notes, and a final-generation gate.
- **How it is implemented:** revisions are recorded with a previous-draft hash; final issue requires `LIVE_GENAI`, a passing deterministic review, and no critic `REVIEW_REQUIRED` verdict.
- **Why it is implemented this way:** the human clinician remains accountable for approval and an offline template cannot be represented as live GenAI-issued content.

In [6]:
revision_history = []
if CLINICIAN_REVISION_INSTRUCTIONS.strip():
    revision_history.append({
        "revision_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "instructions": CLINICIAN_REVISION_INSTRUCTIONS.strip(),
        "previous_draft_hash": hashlib.sha256(draft["text"].encode("utf-8")).hexdigest(),
    })
    draft = generate_draft(confirmed_evidence, retrieved_policy, CERTIFICATE_STYLE, CLINICIAN_REVISION_INSTRUCTIONS.strip(), draft["text"])
    local_draft_review = deterministic_review(draft["text"], confirmed_evidence, ml_payload)
    critic_draft_review = critic_review(draft["text"], confirmed_evidence, retrieved_policy)
    draft_gate = review_gate(local_draft_review, critic_draft_review)


def create_certificate_id():
    return f"EMC-{date.today().strftime('%Y%m%d')}-{uuid.uuid4().hex[:6].upper()}"


def generate_final(evidence, documents, reviewed_draft, certificate_id):
    prompt = f"""CONFIRMED EVIDENCE:
{json.dumps(evidence, indent=2)}

APPROVAL: {json.dumps({"certificate_id": certificate_id, "approval_date": date.today().isoformat(), "approval_notes": APPROVAL_NOTES})}

POLICY:
{policy_context(documents)}

REVIEWED DRAFT:
{reviewed_draft}

Output only the final EMC with headings: ELECTRONIC MEDICAL CERTIFICATE; PATIENT AND CONSULTATION DETAILS; MEDICAL LEAVE CERTIFICATION; ISSUING CLINICIAN AND CLINIC; ELECTRONIC APPROVAL STATEMENT."""
    return call_live_text("You generate a final EMC after documented clinician approval. Use only confirmed evidence and policy.", prompt)

certificate_id = None
final_emc = None
issue_status = "PENDING_REVIEW"
if CLINICIAN_DECISION == "APPROVED":
    if draft["generation_mode"] != "LIVE_GENAI":
        issue_status = "BLOCKED_OFFLINE_TEMPLATE"
    elif not draft_gate["approval_allowed"]:
        issue_status = "BLOCKED_SAFETY_REVIEW"
    else:
        certificate_id = create_certificate_id()
        final_result = generate_final(confirmed_evidence, retrieved_policy, draft["text"], certificate_id)
        if final_result["status"] == "LIVE":
            final_emc = final_result["text"]
            final_gate = review_gate(deterministic_review(final_emc, confirmed_evidence, ml_payload), critic_review(final_emc, confirmed_evidence, retrieved_policy))
            issue_status = "APPROVED_FOR_ISSUE" if final_gate["approval_allowed"] else "BLOCKED_FINAL_SAFETY_REVIEW"
        else:
            issue_status = "BLOCKED_FINAL_GENERATION"

print("Issue status:", issue_status)

Issue status: PENDING_REVIEW


### Step 7: Privacy-Aware Audit and Guardrail Evaluation
- **What is implemented:** a redacted audit record and guardrail test cases for the generation workflow.
- **How it is implemented:** patient identifiers are hashed, draft previews are redacted, and the record captures version, policy sources, generation mode, review status, and text hashes.
- **Why it is implemented this way:** the project can demonstrate traceability without storing API keys or raw patient identifiers in its audit artifact.
- **Production limitation:** deployment would additionally require verified access controls, retention rules, consent governance, and institutional audit policy.

In [7]:
def sha256_text(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def redact_for_audit(text, evidence):
    redacted = text.replace(str(evidence["patient_name"]), "[REDACTED_PATIENT_NAME]")
    redacted = redacted.replace(str(evidence["patient_id"]), "[REDACTED_PATIENT_ID]")
    return redacted[:1000]


audit_record = {
    "audit_version": "v4.0", "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "prompt_version": PROMPT_VERSION, "configured_model": GENAI_MODEL, "live_genai_enabled": bool(client),
    "certificate_id": certificate_id, "issue_status": issue_status,
    "patient_id_hash": sha256_text(confirmed_evidence["patient_id"]),
    "policy_source_ids": [document["id"] for document in retrieved_policy],
    "ml_review_context": {
        "predicted_label": ml_payload["primary_predicted_diagnosis"],
        "model_score_percentage": ml_payload["prediction_confidence_percentage"],
        "requires_manual_review": ml_payload.get("requires_manual_review", True),
    },
    "draft_generation_mode": draft["generation_mode"], "draft_hash": sha256_text(draft["text"]),
    "redacted_draft_preview": redact_for_audit(draft["text"], confirmed_evidence),
    "draft_gate": draft_gate, "revision_history": revision_history,
    "final_hash": sha256_text(final_emc) if final_emc else None,
}
joblib.dump(audit_record, AUDIT_PATH)

baseline_text = offline_template(confirmed_evidence)
leaked_ml_text = baseline_text + "\nModel confidence: 99%"
leaked_diagnosis_text = baseline_text + f"\nDiagnosis: {ml_payload['primary_predicted_diagnosis']}"
checks = []
for name, text, expect_block in [
    ("baseline", baseline_text, False),
    ("ml_leak", leaked_ml_text, True),
    ("diagnosis_without_consent", leaked_diagnosis_text, not confirmed_evidence["diagnosis_disclosure_consent"]),
]:
    actual_block = deterministic_review(text, confirmed_evidence, ml_payload)["verdict"] == "REVIEW_REQUIRED"
    checks.append({"case": name, "expected_block": expect_block, "actual_block": actual_block, "passed": expect_block == actual_block})

guardrail_evaluation = pd.DataFrame(checks)
print(guardrail_evaluation.to_string(index=False))
assert guardrail_evaluation["passed"].all(), "A guardrail test failed."
print(f"Saved privacy-aware audit trail: {AUDIT_PATH.resolve()}")

                     case  expected_block  actual_block  passed
                 baseline           False         False    True
                  ml_leak            True          True    True
diagnosis_without_consent            True          True    True
Saved privacy-aware audit trail: C:\Users\jys20\miniconda3\envs\it3100\AAP\latest_emc_audit_trail.pkl


### Summary
- **Implemented:** environment-based live GenAI configuration, validated ML handoff, policy retrieval, optional note extraction, evidence-grounded drafting, two-layer safety review, clinician approval, and redacted audit records.
- **Implementation approach:** only confirmed facts and retrieved policy enter patient-facing generation; internal ML context remains clinician-only; final issuance requires a fresh live generation and passing safety gate.
- **Why this approach:** it makes GenAI useful for controlled document composition while preventing autonomous diagnosis, invented facts, privacy leakage, and unapproved certificate issuance.
- **Outputs:** pending-review drafts, structured review verdicts, revision history, approved final EMCs, and redacted audit artifacts.
- **Limitation and control:** the demonstration policy corpus is not institutional policy, and every issued EMC remains the responsibility of the approving clinician.